In [ ]:
import json


class Cluster:
    def __init__(self, representative, members):
        self.representative = representative
        self.members = members

    def __repr__(self):
        return (
            f"Cluster(representative='{self.representative}', members={self.members})"
        )


def clean_name(name):
    return name.split("/")[-1].split(".")[0]


def load_clusters(array):
    result = []
    for obj in array:
        representative = clean_name(obj["representative"])
        members = list(map(clean_name, obj["members"]))
        result.append(Cluster(representative, members))
    return result


def load_all_clusters():
    for mode in ["approximate"]:
        for method in [
            "hierarchical",
            "affinity-propagation",
            "facility-location",
        ]:
            path = f"{mode}-{method}.json"
            with open(path) as f:
                data = json.load(f)
                yield (mode, method, load_clusters(data["clustering"]["clusters"]))


clustering = {}
for mode, method, clusters in load_all_clusters():
    clustering[(mode, method)] = clusters

print(f"Loaded {len(clustering)} clustering variants")
for (mode, method), clusters in clustering.items():
    print(f"  {mode} {method}: {len(clusters)} clusters")


Loaded 3 clustering variants
  approximate hierarchical: 336 clusters
  approximate affinity-propagation: 28 clusters
  approximate facility-location: 169 clusters


In [4]:
import pandas as pd

df = pd.read_csv("geometric_features.csv")
print(f"Dataset shape: {df.shape}")
print(f"Positive (double_tetrad=True): {df['double_tetrad'].sum()}")
print(f"Negative (double_tetrad=False): {(~df['double_tetrad']).sum()}")
df.head()


Dataset shape: (2187, 408)
Positive (double_tetrad=True): 1672
Negative (double_tetrad=False): 515


,d01,d02,d03,d04,d05,d06,d07,d12,d13,d14,...,ts3467,ta3467,t3567,ts3567,ta3567,t4567,ts4567,ta4567,source_file,double_tetrad
0,8.662151,11.599962,16.048637,16.380459,14.495792,11.585704,5.262480,5.274370,11.494443,14.490152,...,0.302223,0.953237,-1.113762,-0.897365,0.441289,-1.592939,-0.999755,-0.022141,DT_139d-assembly1_000,True
1,5.827812,11.563400,14.839359,16.342267,15.661431,11.536212,7.674943,7.692990,11.494443,15.659610,...,0.147923,0.988999,0.899351,0.782924,0.622118,1.579374,0.999963,-0.008578,DT_139d-assembly1_001,True
2,9.691375,11.563400,16.340026,16.342267,14.020494,11.536212,4.796926,4.771508,11.474521,14.025507,...,0.510449,0.859908,-1.288887,-0.960526,0.278190,-1.676258,-0.994444,-0.105267,DT_139d-assembly1_002,True
3,6.678694,11.430656,15.389796,15.902359,16.836220,11.277481,9.025119,7.968933,11.391467,14.899982,...,0.159109,0.987261,1.223848,0.940415,0.340030,1.901241,0.945898,-0.324464,DT_143d-assembly1_000,True
4,10.610341,10.842241,15.857459,15.669200,12.846530,11.295434,5.992458,4.637652,11.391467,13.442553,...,0.721439,0.692478,-1.522715,-0.998844,0.048063,-1.732280,-0.986990,-0.160782,DT_143d-assembly1_001,True


In [ ]:
import itertools

N = 8

# Generate column names to drop (raw angle values, because we have the sine and cosine of these angles)
columns_to_drop = []

for i, j, k in itertools.combinations(range(N), 3):
    columns_to_drop.append(f"a{i}{j}{k}")

for i, j, k, l in itertools.combinations(range(N), 4):
    columns_to_drop.append(f"t{i}{j}{k}{l}")

df_filtered = df.drop(columns=columns_to_drop)
print(f"Original columns: {len(df.columns)}")
print(f"Filtered columns: {len(df_filtered.columns)}")
print(
    f"Feature columns: {len(df_filtered.columns) - 2}  (excluding source_file and double_tetrad)"
)


Original columns: 408
Filtered columns: 282
Feature columns: 280  (excluding source_file and double_tetrad)


In [ ]:
from sklearn.model_selection import train_test_split

positive = df_filtered[df_filtered["double_tetrad"]]
negative = df_filtered[~df_filtered["double_tetrad"]]
splits = {}

for (mode, method), clusters in clustering.items():
    clusters.sort(key=lambda c: len(c.members))

    positive_test_names = []

    for cluster in clusters:
        positive_test_names.extend([cluster.representative] + cluster.members)
        if len(positive_test_names) >= 0.25 * len(positive):
            break

    positive_train = positive[~positive["source_file"].isin(positive_test_names)]
    positive_test = positive[positive["source_file"].isin(positive_test_names)]

    negative_train, negative_test = train_test_split(
        negative, test_size=0.25, random_state=42
    )

    df_train = pd.concat([positive_train, negative_train])
    df_test = pd.concat([positive_test, negative_test])

    X_train = df_train.drop(columns=["source_file", "double_tetrad"])
    X_test = df_test.drop(columns=["source_file", "double_tetrad"])
    y_train = df_train["double_tetrad"]
    y_test = df_test["double_tetrad"]
    splits[(mode, method)] = (X_train, y_train, X_test, y_test)
    print(
        f"{mode} {method}: train={len(df_train)} (pos {len(positive_train)}, neg {len(negative_train)}), test={len(df_test)} (pos {len(positive_test)}, neg {len(negative_test)})"
    )


approximate hierarchical: train=1637 (pos 1251, neg 386), test=550 (pos 421, neg 129)
approximate affinity-propagation: train=1579 (pos 1193, neg 386), test=608 (pos 479, neg 129)
approximate facility-location: train=1635 (pos 1249, neg 386), test=552 (pos 423, neg 129)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

classifiers = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42, probability=True),
}


def evaluate_classifier(X_train, y_train, X_test, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, classifier in classifiers.items():
        classifier.fit(X_train_scaled, y_train)
        y_pred = classifier.predict(X_test_scaled)
        yield (
            name,
            classifier,
            scaler,
            classification_report(y_test, y_pred, output_dict=True),
        )


In [9]:
from pickle import dump

N = 8

for (mode, method), (X_train, y_train, X_test, y_test) in splits.items():
    print(f"\n=== Evaluating {mode} {method} ===")
    for name, clf, scaler, report in evaluate_classifier(
        X_train, y_train, X_test, y_test
    ):
        print(f"  Classifier: {name}")
        print(
            f"    accuracy={report['accuracy']:.4f}  "
            f"pos_precision={report['True']['precision']:.4f}  "
            f"pos_recall={report['True']['recall']:.4f}  "
            f"pos_f1={report['True']['f1-score']:.4f}"
        )

        model_name = name.lower().replace(" ", "-")
        payload = {
            "classifier_name": name,
            "classifier": clf,
            "scaler": scaler,
            "feature_columns": list(X_train.columns),
            "window_size": N,
            "positive_label": True,
            "split_mode": mode,
            "split_method": method,
        }

        with open(f"{mode}-{method}-{model_name}.pkl", "wb") as f:
            dump(payload, f)

print("\nDone. Trained and saved 5 classifiers x 3 splits = 15 model bundles.")



=== Evaluating approximate hierarchical ===
  Classifier: Naive Bayes
    accuracy=0.8582  pos_precision=1.0000  pos_recall=0.8147  pos_f1=0.8979
  Classifier: Logistic Regression
    accuracy=0.9727  pos_precision=0.9951  pos_recall=0.9691  pos_f1=0.9819
  Classifier: Decision Tree
    accuracy=0.9036  pos_precision=0.9920  pos_recall=0.8812  pos_f1=0.9333
  Classifier: Random Forest
    accuracy=0.9473  pos_precision=1.0000  pos_recall=0.9311  pos_f1=0.9643
  Classifier: SVM
    accuracy=0.9745  pos_precision=1.0000  pos_recall=0.9667  pos_f1=0.9831

=== Evaluating approximate affinity-propagation ===
  Classifier: Naive Bayes
    accuracy=0.7385  pos_precision=1.0000  pos_recall=0.6681  pos_f1=0.8010
  Classifier: Logistic Regression
    accuracy=0.9359  pos_precision=0.9977  pos_recall=0.9207  pos_f1=0.9577
  Classifier: Decision Tree
    accuracy=0.9688  pos_precision=0.9832  pos_recall=0.9770  pos_f1=0.9801
  Classifier: Random Forest
    accuracy=0.9720  pos_precision=1.0000  p